# 03 — Benchmark Collection (SPY)

## Purpose
Collect the market benchmark that the project needs for **beta** (how much a stock follows the market) and **abnormal returns** (the part of a stock's move that the whole market does not explain).

**SPY** is a fund that tracks the S&P 500, the 500 largest US companies. Its price stands for "the market".

## Rules this notebook enforces
1. **Same price convention as our stock file.** `market_clean.csv` holds dividend-adjusted prices (AAPL closes at 125.63 on 2021-01-04, while the unadjusted close was about 129.4). SPY must use adjusted prices too. The notebook proves this by downloading AAPL again and comparing it with `market_clean.csv`.
2. **Same dates.** SPY must have exactly the dates of `market_clean.csv` (2021-01-04 to 2025-12-30, 1,254 trading days). We do **not** pull 2025-12-31: re-pulling would force every other notebook to be re-run, so the data cutoff is 2025-12-30 and is stated in the report.
3. **Same return definition.** `return = (close_t - close_(t-1)) / close_(t-1)`, empty on the first day.

## Outputs
| File | Columns | Used by |
|---|---|---|
| `data/raw/SPY.csv` | `date, close` | Person 3's risk engine (`benchmark=...` in `04_risk_engine.ipynb`) |
| `data/processed/benchmark_prices.csv` | `symbol, date, close, return` | the warehouse (`staging.benchmark_prices`), event study, machine learning |

## Install (once)
In a terminal with the project environment active: `pip install yfinance`. If Yahoo is not reachable from your computer, use the manual option in section 2.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")
MARKET_PATH = PROCESSED_DIR / "market_clean.csv"
SPY_RAW_PATH = RAW_DIR / "SPY.csv"
BENCH_PATH = PROCESSED_DIR / "benchmark_prices.csv"

SYMBOL = "SPY"
START = "2021-01-04"
LAST_DAY = "2025-12-30"                                   # last day we keep (inclusive)
END_EXCLUSIVE = (pd.Timestamp(LAST_DAY) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")   # Yahoo's end date is exclusive

# Option: if Yahoo cannot be reached, download the SPY "Historical data" CSV by hand from finance.yahoo.com
# (period 2021-01-04 to 2025-12-30, frequency daily) and put its path here, for example Path("SPY_manual.csv").
MANUAL_CSV = None

market = pd.read_csv(MARKET_PATH, usecols=["ticker", "date", "close"], parse_dates=["date"])
market_days = pd.DatetimeIndex(sorted(market["date"].unique()))
print(f"market_clean.csv: {len(market_days):,} trading days, {market_days.min().date()} to {market_days.max().date()}")
assert market_days.min() == pd.Timestamp(START) and market_days.max() == pd.Timestamp(LAST_DAY)

market_clean.csv: 1,254 trading days, 2021-01-04 to 2025-12-30


## 1. Download helper

`auto_adjust=True` makes Yahoo return the **adjusted** close in the `Close` column (splits and dividends applied), the same convention as our stock file. Newer versions of `yfinance` return two-level column names; the helper flattens them.

In [2]:
def get_adjusted_close(symbol):
    """Return a Series (index = date, values = adjusted close) for `symbol` between START and LAST_DAY."""
    if MANUAL_CSV is not None and symbol == SYMBOL:
        raw = pd.read_csv(MANUAL_CSV)
        raw.columns = [str(c).strip() for c in raw.columns]
        price_col = "Adj Close" if "Adj Close" in raw.columns else "Close"
        print(f"Manual file: using column '{price_col}'")
        s = pd.Series(raw[price_col].to_numpy(dtype=float), index=pd.to_datetime(raw["Date"]), name="close")
    else:
        import yfinance as yf
        data = yf.download(symbol, start=START, end=END_EXCLUSIVE, auto_adjust=True, progress=False)
        if data is None or len(data) == 0:
            raise RuntimeError(f"Yahoo returned no data for {symbol}. Check the internet connection or use MANUAL_CSV.")
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        s = data["Close"].astype(float).rename("close")
        s.index = pd.to_datetime(s.index)
    s.index = s.index.tz_localize(None) if s.index.tz is not None else s.index
    s.index.name = "date"
    s = s.sort_index()
    s = s[(s.index >= pd.Timestamp(START)) & (s.index <= pd.Timestamp(LAST_DAY))]
    return s[~s.index.duplicated(keep="last")]

## 2. Download SPY

In [3]:
spy = get_adjusted_close(SYMBOL)
print(f"SPY: {len(spy):,} rows, {spy.index.min().date()} to {spy.index.max().date()}")
display(spy.head(3).to_frame())
display(spy.tail(3).to_frame())

SPY: 1,254 rows, 2021-01-04 to 2025-12-30


,close
date,
2021-01-04,341.588745
2021-01-05,343.941284
2021-01-06,345.997559


,close
date,
2025-12-26,684.959961
2025-12-29,682.519043
2025-12-30,681.685547


## 3. Prove the price convention

Download AAPL the same way and compare it with `market_clean.csv`. If our file is adjusted and SPY is adjusted too, the ratio is about 1.000 on every day. An unadjusted download would be about 3% higher on 2021-01-04 for AAPL and the check would stop.

In [4]:
aapl_new = get_adjusted_close("AAPL") if MANUAL_CSV is None else None
if aapl_new is None:
    print("Manual SPY file used: the AAPL convention check is skipped. Check by hand that the SPY close you use is the 'Adj Close' column.")
else:
    aapl_old = market[market["ticker"] == "AAPL"].set_index("date")["close"]
    both = pd.concat([aapl_old.rename("ours"), aapl_new.rename("download")], axis=1).dropna()
    ratio = both["download"] / both["ours"]
    print(f"AAPL 2021-01-04: ours {both['ours'].iloc[0]:.4f}, download {both['download'].iloc[0]:.4f}")
    print(f"Ratio download / ours: min {ratio.min():.4f}, max {ratio.max():.4f}")
    assert ratio.sub(1).abs().max() < 0.01, "The download uses a different price convention than market_clean.csv"
    print("Convention check passed: same adjusted prices as our stock file.")

AAPL 2021-01-04: ours 125.6325, download 125.5371
Ratio download / ours: min 0.9992, max 0.9992
Convention check passed: same adjusted prices as our stock file.


## 4. Check SPY against our trading calendar

In [5]:
spy_days = pd.DatetimeIndex(spy.index)
only_spy = spy_days.difference(market_days)
only_market = market_days.difference(spy_days)
print("Days only in SPY:   ", [str(d.date()) for d in only_spy][:10], f"({len(only_spy)})")
print("Days only in market:", [str(d.date()) for d in only_market][:10], f"({len(only_market)})")
assert len(only_spy) == 0 and len(only_market) == 0, "SPY dates differ from market_clean.csv"
assert spy.notna().all() and (spy > 0).all()
print(f"SPY has exactly the same {len(spy):,} dates as market_clean.csv.")

Days only in SPY:    [] (0)
Days only in market: [] (0)
SPY has exactly the same 1,254 dates as market_clean.csv.


## 5. Build the two output files

`return` is computed exactly like in `market_clean.csv` (simple daily percentage change of the close, empty on the first day). Rounding follows that file: 4 decimals for prices, 6 for returns.

In [6]:
bench = spy.rename("close").reset_index()
bench["return"] = bench["close"].pct_change()
bench["close"] = bench["close"].round(4)
bench["return"] = bench["return"].round(6)
bench.insert(0, "symbol", SYMBOL)
bench["date"] = bench["date"].dt.strftime("%Y-%m-%d")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
bench[["date", "close"]].to_csv(SPY_RAW_PATH, index=False)
bench[["symbol", "date", "close", "return"]].to_csv(BENCH_PATH, index=False)
print("Saved:", SPY_RAW_PATH.resolve())
print("Saved:", BENCH_PATH.resolve())
display(bench.head())

Saved: D:\DEPI\MarketPulse Graduation Project\data\raw\SPY.csv
Saved: D:\DEPI\MarketPulse Graduation Project\data\processed\benchmark_prices.csv


,symbol,date,close,return
0,SPY,2021-01-04,341.5887,NaN
1,SPY,2021-01-05,343.9413,0.006887
2,SPY,2021-01-06,345.9976,0.005979
3,SPY,2021-01-07,351.1382,0.014857
4,SPY,2021-01-08,353.1389,0.005698


## 6. Sanity checks

A benchmark should look like "the market": positive but not perfect correlation with each stock, a deep fall in 2022, a higher price at the end.

In [7]:
checks = {
    "1,254 rows": len(bench) == len(market_days),
    "first return is empty, nothing else is": bool(bench["return"].isna().sum() == 1 and pd.isna(bench["return"].iloc[0])),
    "no duplicate dates": bool(bench["date"].is_unique),
    "dates sorted": bool(bench["date"].is_monotonic_increasing),
    "largest daily move under 15%": bool(bench["return"].abs().max() < 0.15),
}

stock_ret = market.sort_values(["ticker", "date"]).assign(ret=lambda d: d.groupby("ticker")["close"].pct_change()).pivot(index="date", columns="ticker", values="ret")
spy_ret = pd.Series(bench["return"].to_numpy(), index=pd.to_datetime(bench["date"]))
corr = stock_ret.corrwith(spy_ret).round(2).sort_values(ascending=False)
display(corr.rename("correlation with SPY").to_frame())
checks["every stock correlates positively with SPY (above 0.1)"] = bool((corr > 0.1).all())

spy_close = pd.Series(bench["close"].to_numpy(), index=pd.to_datetime(bench["date"]))
drawdown = spy_close / spy_close.cummax() - 1
print(f"SPY total return {spy_close.iloc[-1] / spy_close.iloc[0] - 1:.1%}, worst drawdown {drawdown.min():.1%} on {drawdown.idxmin().date()}")
checks["SPY worst drawdown happened in 2022 and is between -35% and -15%"] = bool(drawdown.idxmin().year == 2022 and -0.35 < drawdown.min() < -0.15)

result = pd.Series(checks, name="passed")
display(result.to_frame())
assert result.all(), "At least one check failed. Read the table above before using the file."
print("All checks passed. Send SPY.csv and benchmark_prices.csv to Person 3 and Person 4.")

,correlation with SPY
ticker,
AAPL,0.76
MSFT,0.76
AMZN,0.72
NVDA,0.70
JPM,0.62
TSLA,0.57
KO,0.34
XOM,0.34
LLY,0.31


SPY total return 99.6%, worst drawdown -24.5% on 2022-10-12


,passed
"1,254 rows",True
"first return is empty, nothing else is",True
no duplicate dates,True
dates sorted,True
largest daily move under 15%,True
every stock correlates positively with SPY (above 0.1),True
SPY worst drawdown happened in 2022 and is between -35% and -15%,True


All checks passed. Send SPY.csv and benchmark_prices.csv to Person 3 and Person 4.


## Completion checklist

- [ ] The convention check (section 3) passed, or the manual file's `Adj Close` column was used.
- [ ] SPY has exactly the 1,254 dates of `market_clean.csv`.
- [ ] `data/raw/SPY.csv` (`date, close`) sent to Person 3; `data/processed/benchmark_prices.csv` sent to Person 4 and Person 1.
- [ ] The data cutoff (2025-12-30) is written in the report.